# Setup (Dependencies & Imports)

Install dependencies, clone the repo, and import required libraries and modules.

In [ ]:
# Install dependencies
!pip install -q accelerate bitsandbytes


In [ ]:
# Clone the project repo
!git clone https://github.com/gizayceylan/FakeNews.git

# Add it to Python path
import sys
sys.path.append("/content/FakeNews")


In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import torch
import requests
import json

from numpy.linalg import norm
from PIL import Image
from tqdm import tqdm
from google.colab import files

# ML/DL Libraries
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from torchvision.datasets.utils import download_url


# Metadata (Dataset Loading & Preparation)

Load the Fakeddit metadata, clean it, and select the subset of samples containing valid image URLs.

In [ ]:
# Load metadata
dataset_url = "https://raw.githubusercontent.com/gizayceylan/FakeNews/main/Datasets/fakeddit_small_2_way.csv"
df = pd.read_csv(dataset_url)
print("Shape original:", df.shape)

# Remove duplicate rows (if any)
df = df.drop_duplicates(keep="first")
print("Shape after removing duplicates:", df.shape)

# Keep entries with valid image URLs
df = df[df["image_url"].notna()].reset_index(drop=True)
print("Shape after removing rows without image_url:", df.shape)

df.head()

In [ ]:
# Check label distribution
label_counts = df['2_way_label'].value_counts().sort_index()
label_percent = df['2_way_label'].value_counts(normalize=True).sort_index() * 100

# Combine into one table for display
balance_df = pd.DataFrame({
    'Label': ['Fake (0)', 'Real (1)'],
    'Count': label_counts.values,
    'Percentage': label_percent.values.round(2)
})

balance_df

In [ ]:
# Make a folder for images
img_dir = "/content/fakeddit_images"
os.makedirs(img_dir, exist_ok=True)


In [ ]:
# Sample N images for testing
N = 300
sample_df = df.sample(N, random_state=42).reset_index(drop=True)


In [ ]:
# Download images from URLs
image_paths = []
failed = 0

for i, url in tqdm(list(enumerate(sample_df["image_url"])), total=len(sample_df)):
    filename = f"{i}.jpg"          # index inside sample
    filepath = os.path.join(img_dir, filename)

    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            with open(filepath, "wb") as f:
                f.write(r.content)
            image_paths.append(filepath)
        else:
            image_paths.append(None)
            failed += 1
    except Exception:
        image_paths.append(None)
        failed += 1

sample_df["image_path"] = image_paths
print("\nFailed downloads:", failed)


In [ ]:
# Keep only successfully downloaded images
image_df = sample_df[sample_df["image_path"].notna()].reset_index(drop=True)
image_df = image_df[['image_path', 'clean_title','2_way_label', 'image_url']].rename(columns={'2_way_label': 'label'})
print("Images available:", len(image_df))
image_df.head()


In [ ]:
# Test a sample
Image.open(image_df["image_path"].iloc[0])


# BLIP-2

Extract semantic descriptions of the images.

In [ ]:
# Check for GPU (Crucial for BLIP-2)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Load BLIP-2 model & processor
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16,
    device_map="auto" # Automatically maps to GPU
)


In [ ]:
# Generate captions
def generate_blip2_caption(image_path):
    """
    Generates a detailed caption using BLIP-2.
    """
    try:
        image = Image.open(image_path).convert("RGB")

        # BLIP-2 doesn't need a specific prompt to start, but you can add one if needed.
        # For pure captioning, we just pass the image.
        inputs = processor(images=image, return_tensors="pt").to(device, torch.float16)

        # Generate
        generated_ids = model.generate(**inputs, max_new_tokens=50)
        caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

        return caption
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
print(f"\nStarting generation for {len(image_df)} images...")
blip2_captions = []

for i, row in tqdm(image_df.iterrows(), total=len(image_df)):
    cap = generate_blip2_caption(row['image_path'])
    blip2_captions.append(cap)

# Add to DataFrame
image_df['blip2_caption'] = blip2_captions

# Show example
print("\nExample Caption:")
print(f"Image: {image_df.iloc[0]['image_path']}")
print(f"Caption: {image_df.iloc[0]['blip2_caption']}")

In [ ]:
image_df.head()

In [ ]:
# Save the final CSV with the new 'blip2_caption' column
output_file = "fakeddit_subset_blip2captions.csv"
image_df.to_csv(output_file, index=False)

# Download the CSV
files.download(output_file)